# 82514 · Sesión S18 — Bayes y Kalman: la puerta, el pasillo y la ganancia

**Bloque 6** · jueves 19 de noviembre de 2026 · 2 h · IQS Universitat Ramon Llull

La regla de Bayes con la puerta estrecha, el filtro de Bayes en el pasillo junto al muro y Kalman en una dimensión.

| | |
|---|---|
| **En clase** | Cinco minutos, por parejas: las secciones 1 a 3, un número en cada una, y «✎ Vuestra respuesta». Al acabar, abrid el S18b. |
| **En casa** | Sección 4 y ejercicios 1 a 3; las soluciones, al final. |
| **Entrega** | Al acabar la clase, individual: este cuaderno y el S18b, ejecutados y con «✎ Vuestra respuesta» escrita (Archivo → Descargar → Descargar .ipynb), a la Tarea «Taller S18: estimación y localización» de Moodle. Sin nota: presentado o no presentado. |

Cada paso tiene tres celdas: **▶ Parámetros**, lo único que se cambia; **⚙ Cálculo**, que no hace falta tocar; y **✔ Resultado**, lo que sale.

---

In [ ]:
# @title ⚙ Instalación · ejecutad esta celda primero
# Ejecutar una sola vez. En Colab tarda unos segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('matplotlib', 'matplotlib')])

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

VERSIONES = {'numpy': '2.4.6', 'matplotlib': '3.10.0'}   # las de todo el bloque 6 (las de Colab al prepararlo)
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.4)
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.axisbelow'] = True
# los colores de todo el cuaderno: la verdad, en verde; lo estimado (la creencia, Kalman), en azul; la odometría sola,
# en rojo; la medida, en ámbar; lo predicho y las referencias, en gris
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def cien(v, n=2):
    """Un número pequeño o grande con coma decimal y potencia de diez: 1,40·10⁻¹⁰."""
    m, e = f'{v:.{n}e}'.split('e')
    return m.replace('.', ',').replace('-', '−') + '·10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

print('numpy', np.__version__, '· matplotlib', matplotlib.__version__, '· en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy', VERSIONES['numpy'], '; las cifras con semilla pueden cambiar un poco.')

## 1. ¿Está abierta la puerta estrecha?

*En clase · diapositivas 47 a 51*

La regla de Bayes con el detector de la tabla: «abierta» con 0,6 si lo está y con 0,2 si está cerrada.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def bayes_puerta(b, z, acierto=0.6, falsa=0.2):
    """Una medida del detector de la puerta estrecha, con la regla de Bayes.
    b: p(abierta) antes de la medida. z: lo que dice el detector, 'abierta' o 'cerrada'.
    acierto = p(«abierta» | abierta); falsa = p(«abierta» | cerrada), la falsa alarma.
    Devuelve p(abierta) despues de la medida y el normalizador η = 1/p(z)."""
    if z == 'abierta':
        p_si, p_no = acierto, falsa              # p(z | abierta) y p(z | cerrada)
    else:
        p_si, p_no = 1 - acierto, 1 - falsa
    eta = 1 / (p_si * b + p_no * (1 - b))       # 1 / p(z): la probabilidad total de la medida
    return eta * p_si * b, eta


def secuencia(medidas, acierto=0.6, falsa=0.2, b=0.5):
    """La creencia tras cada medida, encadenadas: la de despues de una es la de antes de la siguiente."""
    creencias = [b]
    for z in medidas:
        b, _ = bayes_puerta(b, z, acierto, falsa)
        creencias.append(b)
    return creencias

B1, ETA = bayes_puerta(0.5, 'abierta')                      # una «abierta», desde 0,5
TRES = secuencia(['abierta'] * 3)                           # tres «abierta» seguidas
ABIERTA_CERRADA = secuencia(['abierta', 'cerrada'])         # «abierta» y luego «cerrada»
NO_INFORMA = secuencia(['abierta', 'abierta'], 0.5, 0.5)    # para la figura: un detector de 0,5 y 0,5

In [ ]:
# @title ✔ Resultado
print(f'p(«abierta») = 0,6·0,5 + 0,2·0,5 = 0,4  ->  η = {coma(ETA)} ;  p(abierta | «abierta») = {coma(B1)}')
print('Tres «abierta» seguidas:   ', ' -> '.join(coma(v, 3) for v in TRES))
print('«abierta» y luego «cerrada»:', ' -> '.join(coma(v, 3) for v in ABIERTA_CERRADA))

fig, ax = plt.subplots(figsize=(7, 3.3))
x_ = np.arange(len(TRES))
ax.bar(x_, TRES, width=0.55, color=IQS_AZUL, label='«abierta», «abierta», «abierta»')
for i, v in enumerate(TRES):
    ax.text(i, v + 0.03, coma(v, 3 if i == 3 else 2), ha='center', fontsize=10, color=IQS_AZUL)
ax.bar(2.32, ABIERTA_CERRADA[2], width=0.22, color=AMBAR, label='la 2.ª, «cerrada»')
ax.text(2.46, ABIERTA_CERRADA[2] + 0.03, coma(ABIERTA_CERRADA[2], 1), ha='left', fontsize=10, color=AMBAR)
ax.plot(x_[:3], NO_INFORMA, 'o--', color=GRIS, lw=1.5, ms=6, label='un detector de 0,5 y 0,5')
ax.set_xticks(x_, ['sin mirar', '1.ª medida', '2.ª', '3.ª']); ax.set_xlim(-0.5, 3.6)
ax.set_ylim(0, 1.42); ax.set_yticks([0, 0.25, 0.5, 0.75, 1]); ax.yaxis.set_major_formatter(COMA); ax.grid(axis='x')
ax.set_ylabel('p(abierta)'); ax.legend(loc='upper left', ncol=2, fontsize=9, frameon=False)
ax.set_title('La creencia tras cada medida, con el detector de la tabla (0,6 y 0,2)', fontsize=11)
plt.show()

**Qué tiene que salir:** η = 2,50 y 0,75; tres «abierta», 0,964; «abierta» y «cerrada», 0,600.

### En clase: hasta pasar de 0,99

Subid `N` hasta pasar de 0,99, con el detector de la tabla y con uno de 0,95 y 0,05.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el detector de la puerta: p(«abierta» | abierta) y p(«abierta» | cerrada), la falsa alarma [de 0 a 1]
# probad también 0.95 y 0.05
ACIERTO = 0.6  # @param {type:"number"}
FALSA = 0.2  # @param {type:"number"}
# cuántas «abierta» seguidas [medidas]: subidlo hasta pasar de 0,99
N = 2  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
B_N = secuencia(['abierta'] * int(N), ACIERTO, FALSA)      # la creencia sin mirar y tras cada «abierta»
PASA = B_N[-1] > 0.99
PRIMERA = next((i for i, v in enumerate(B_N) if v > 0.99), None)   # la primera «abierta» que pasa de 0,99 (None: ninguna)
# para la figura: lo que le falta a la creencia hasta 1, 1 − p(abierta), y el detector de la tabla, si el vuestro es otro.
# Sin restar a 1 (se quedaría en 1,1·10⁻¹⁶): desde 0,5 y con k «abierta», 1 − p = falsaᵏ / (aciertoᵏ + falsaᵏ)
K_N = np.arange(int(N) + 1)
FALTA_N = FALSA**K_N / (ACIERTO**K_N + FALSA**K_N)
B_TABLA = None if (ACIERTO, FALSA) == (0.6, 0.2) else secuencia(['abierta'] * int(N))
FALTA_TABLA = None if B_TABLA is None else 0.2**K_N / (0.6**K_N + 0.2**K_N)

In [ ]:
# @title ✔ Resultado
print(f'Detector {coma(ACIERTO)} / {coma(FALSA)}, {N} «abierta»: ' + ' -> '.join(coma(v, 3) for v in B_N)
      + ('   (pasa de 0,99)' if PASA else '   (aún no pasa de 0,99)'))

from matplotlib.patches import Patch
n_, x_ = len(B_N), np.arange(len(B_N))
pasa_ = np.array(B_N) > 0.99                       # las que pasan de 0,99, en azul; las que no, en azul claro
claro_ = matplotlib.colors.to_rgba(IQS_AZUL, 0.4)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(min(15, max(10, 4.5 + 0.9 * n_)), 3.4),
                             gridspec_kw=dict(width_ratios=[1.5, 1], wspace=0.3))
a1.bar(x_, B_N, width=0.6, color=[IQS_AZUL if p else claro_ for p in pasa_])
for i, v in enumerate(B_N):                        # la cifra, dentro de la barra
    a1.text(i, v - 0.02, coma(v, 3), ha='center', va='top', fontsize=9, color='w' if pasa_[i] else IQS_AZUL,
            rotation=0 if n_ <= 11 else 90)
a1.axhline(0.99, color='k', ls='--', lw=1)
a1.text(1.01, 0.99, '0,99', transform=a1.get_yaxis_transform(), ha='left', va='center', fontsize=9.5)
a1.set_xticks(x_); a1.set_xlabel('«abierta» seguidas (0: sin mirar)')
a1.set_xlim(-0.5, n_ - 0.5); a1.set_ylim(0, 1.3); a1.set_yticks([0, 0.25, 0.5, 0.75]); a1.yaxis.set_major_formatter(COMA)
a1.grid(axis='x'); a1.set_ylabel('p(abierta)')
a1.legend(handles=[Patch(color=IQS_AZUL, label='pasa de 0,99'), Patch(color=claro_, label='aún no')],
          loc='upper left', ncol=2, fontsize=9, frameon=False)
a1.set_title(f'Detector {coma(ACIERTO)} y {coma(FALSA)}: ' + (f'pasa de 0,99 en la {PRIMERA}.ª «abierta»'
             if PRIMERA is not None else f'con {int(N)}, aún no pasa de 0,99'), fontsize=11)
if FALTA_TABLA is not None:
    a2.semilogy(x_, FALTA_TABLA, 'o-', color=GRIS, lw=1.5, ms=5, label='el de la tabla: 0,6 y 0,2')
a2.semilogy(x_, FALTA_N, '-', color=IQS_AZUL, lw=1.5, label=f'vuestro: {coma(ACIERTO)} y {coma(FALSA)}')
a2.scatter(x_, FALTA_N, s=40, c=[IQS_AZUL if p else claro_ for p in pasa_], edgecolors=IQS_AZUL, zorder=3)
a2.axhline(0.01, color='k', ls='--', lw=1)
a2.text(1.01, 0.01, '0,01', transform=a2.get_yaxis_transform(), ha='left', va='center', fontsize=9.5)
a2.set_xticks(x_); a2.set_xlim(-0.5, n_ - 0.5); a2.set_xlabel('«abierta» seguidas')
a2.set_ylim(min(FALTA_N.min(), 0.01, 1 if FALTA_TABLA is None else FALTA_TABLA.min()) / 4, 1.5)
a2.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',') if v >= 1e-3 else cien(v, 0)[2:]))
a2.yaxis.set_minor_locator(plt.NullLocator()); a2.set_ylabel('1 − p(abierta)')
a2.set_title('Lo que falta hasta 1, en escala logarítmica', fontsize=11)
if FALTA_TABLA is not None:
    a2.legend(loc='lower left', fontsize=9, frameon=False)
plt.show()

**Qué tiene que salir:** con N = 2, 0,750 y 0,900: aún no pasa.

## 2. El filtro de Bayes en el pasillo junto al muro

*En clase · diapositivas 52 y 53*

El pasillo, en 130 celdas de 10 cm. El detector lateral dice «hueco» con 0,8 frente a una puerta y con 0,1 frente al muro.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
NC = 130                                         # el pasillo de y = 5,2 m: de x = 0 a 13 m, en celdas de 10 cm
XC = (np.arange(NC) + 0.5) * 0.1                 # el centro de cada celda
HUECO = ((XC >= 0.8) & (XC < 2.8)) | ((XC >= 9.2) & (XC < 10.0))   # frente a la puerta ancha o a la estrecha
ANCHA, ESTRECHA = (XC >= 0.8) & (XC < 2.8), (XC >= 9.2) & (XC < 10.0)


def corregir(bel, hueco, acierto=0.8, falsa=0.1):
    """Corregir con lo que dice el detector lateral («hueco» o «pared»): verosimilitud por la creencia, y normalizar."""
    p = np.where(HUECO, acierto, falsa)          # p(«hueco» | celda)
    ver = p if hueco else 1 - p
    bel = ver * bel
    return bel / bel.sum()


def predecir(bel, celdas=5, nucleo=(0.1, 0.8, 0.1)):
    """Predecir el avance de 0,5 m (5 celdas) con su ruido: 4, 5 o 6 celdas, con 0,1, 0,8 y 0,1.
    Es la suma de la probabilidad total: cada celda reparte su creencia entre las tres de destino."""
    nueva = np.zeros_like(bel)
    for desp, w in zip((celdas - 1, celdas, celdas + 1), nucleo):
        nueva[desp:] += w * bel[:NC - desp]      # lo que saldria del pasillo se pierde
    return nueva / nueva.sum()


def recorre(acierto=0.8, falsa=0.1, semilla=3, x0=0.45, pasos=20):
    """El robot recorre el pasillo desde x0, en pasos de 0,5 m, sin saber donde empieza (creencia plana).
    En cada sitio mira de lado y el detector dice «hueco» o «pared» (con sus errores). Devuelve, en cada paso,
    la x verdadera, la medida y la creencia ya corregida."""
    rng = np.random.default_rng(semilla)
    bel, x, hist = np.full(NC, 1 / NC), x0, []
    for k in range(pasos):
        hay = bool(HUECO[int(x / 0.1)])
        z = rng.random() < (acierto if hay else falsa)
        bel = corregir(bel, z, acierto, falsa)
        hist.append((x, z, bel.copy()))
        bel = predecir(bel)
        x += 0.5
    return hist


def masa_cerca(bel, x, d=0.3):
    """La creencia a ±d de la x verdadera."""
    return bel[np.abs(XC - x) <= d + 1e-9].sum()

PLANA = np.full(NC, 1 / NC)                      # sin mirar
bel = corregir(PLANA, True)                      # tras el primer «hueco»
P_ANCHA, P_ESTRECHA = bel[ANCHA].sum(), bel[ESTRECHA].sum()
# para las figuras: la verosimilitud de «hueco», la creencia tras avanzar 0,5 m, y el pasillo (el muro y sus huecos)
VEROSIMIL = np.where(HUECO, 0.8, 0.1)
BEL_PRED = predecir(bel)
BORDES = np.arange(NC + 1) * 0.1                 # los bordes de las celdas [m]
PUERTAS = ((0.8, 2.8, 'puerta ancha'), (9.2, 10.0, 'estrecha'))   # los dos huecos del muro [m]
MURO = ((0.0, 0.8), (2.8, 9.2), (10.0, 13.0))                          # y el muro, entre ellos [m]


def pasillo(ax, rotulos=False):
    """Para las figuras: el muro, en gris oscuro sobre el eje, con sus dos huecos, en gris claro (y sus rótulos)."""
    for a, b_, nom in PUERTAS:
        ax.axvspan(a, b_, color=GRIS, alpha=0.22, lw=0)
        if rotulos:
            ax.text((a + b_) / 2, 1.04, nom, transform=ax.get_xaxis_transform(), ha='center', va='bottom', fontsize=9, color='#555')
    for a, b_ in MURO:
        ax.plot([a, b_], [0, 0], color='#555', lw=4, solid_capstyle='butt', clip_on=False, zorder=4)
    if rotulos:
        ax.text(6.0, 1.04, 'el muro', transform=ax.get_xaxis_transform(), ha='center', va='bottom', fontsize=9, color='#555')
    ax.set_xlim(0, 13); ax.grid(False)
    ax.yaxis.set_major_locator(plt.MaxNLocator(2)); ax.yaxis.set_major_formatter(COMA); ax.tick_params(axis='y', labelsize=9)

In [ ]:
# @title ✔ Resultado
print(f'{NC} celdas; frente a un hueco: {HUECO.sum()} ({ANCHA.sum()} la ancha y {ESTRECHA.sum()} la estrecha)')
print(f'Primera medida «hueco» desde la creencia plana: la ancha {coma(100*P_ANCHA,1)} %, la estrecha {coma(100*P_ESTRECHA,1)} %')

fig, axs = plt.subplots(3, 1, figsize=(10, 4.8), sharex=True, gridspec_kw=dict(hspace=0.5, top=0.89, bottom=0.11))
axs[0].stairs(VEROSIMIL, BORDES, fill=True, color=AMBAR, alpha=0.8)
axs[0].set_title('La verosimilitud de «hueco»: 0,8 frente a las puertas y 0,1 frente al muro', fontsize=10, loc='left', pad=16)
axs[1].stairs(bel, BORDES, fill=True, color=IQS_AZUL)
axs[1].set_title(f'Desde plana, tras un «hueco»: la ancha {coma(100*P_ANCHA,0)} % y la estrecha {coma(100*P_ESTRECHA,0)} %', fontsize=10, loc='left')
axs[2].stairs(BEL_PRED, BORDES, fill=True, color=GRIS, label='predicha')
axs[2].stairs(bel, BORDES, color=IQS_AZUL, lw=1.2, label='antes de predecir')
axs[2].set_title('Y tras predecir 0,5 m: se desplaza y se ensancha', fontsize=10, loc='left')
axs[2].legend(loc='center', bbox_to_anchor=(0.46, 0.55), fontsize=9, frameon=False, ncol=2)
for ax in axs:
    pasillo(ax, rotulos=ax is axs[0])
    ax.set_ylim(0, ax.get_ylim()[1] * 1.05); ax.set_ylabel('p', fontsize=9)
axs[0].set_yticks([0.1, 0.8])
axs[-1].set_xlabel('x a lo largo del pasillo (m)'); ejes(axs[-1]); axs[-1].set_xticks(range(0, 14))
plt.show()

**Qué tiene que salir:** 28 celdas de hueco; tras un «hueco», la ancha 49,1 % y la estrecha 19,6 %.

### En clase: desde x = 5,95 m

Poned `X0 = 5.95`, frente al muro: ¿cuándo queda un solo pico, y por qué tarda más que desde la ancha?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el detector lateral: p(«hueco» | hueco) y p(«hueco» | pared) [de 0 a 1]
DET = (0.8, 0.1)  # @param {type:"raw"}
# de dónde sale el robot [m]: probad 5.95, frente al muro
X0 = 0.95  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
PASOS = int((13 - X0) / 0.5)       # hasta el final del pasillo, en pasos de 0,5 m
H = recorre(*DET, semilla=18, x0=X0, pasos=PASOS)
# en cada paso: la x verdadera, la medida, la creencia a ±0,3 m de la verdad y la de cada puerta
TABLA_H = [(x, z, masa_cerca(b, x), b[ANCHA].sum(), b[ESTRECHA].sum()) for x, z, b in H]
# para la figura: sin mirar, el primer «hueco», cuando pasa del 70 % cerca de la verdad, y tras la estrecha (x > 9,9 m)
primer_hueco = next((i for i, (x, z, b) in enumerate(H) if z), 0)
pasa_70 = next((i for i, (x, z, b) in enumerate(H) if masa_cerca(b, x) > 0.7), len(H) - 1)
TRAS_ESTRECHA = next((i for i, (x, z, b) in enumerate(H) if x > 9.9), len(H) - 1)
PANELES = [None] + sorted({primer_hueco, pasa_70, TRAS_ESTRECHA})
# y la creencia cerca de la verdad, paso a paso, desde X0 y, si es otra, desde la ancha (la diapositiva 53)
CERCA = np.array([(x, 100 * m, z) for x, z, m, _, _ in TABLA_H])
CERCA_ANCHA = None if X0 == 0.95 else np.array(
    [(x, 100 * masa_cerca(b, x), z) for x, z, b in recorre(*DET, semilla=18, x0=0.95, pasos=int((13 - 0.95) / 0.5))])

In [ ]:
# @title ✔ Resultado
for x, z, m, an, es in TABLA_H:
    print(f'x = {coma(x)} m  {"«hueco»" if z else "«pared»"}   a ±0,3 m: {coma(100*m,1):>5} %'
          f'   ancha {coma(100*an,1):>5} %   estrecha {coma(100*es,1):>5} %')

from matplotlib.lines import Line2D
fig, axs = plt.subplots(len(PANELES) + 1, 1, figsize=(10, 1.05 * len(PANELES) + 2.6), sharex=True,
                        gridspec_kw=dict(height_ratios=[1] * len(PANELES) + [2], hspace=0.5, top=0.93, bottom=0.08))
for ax, i in zip(axs, PANELES):
    if i is None:
        b_, t_ = PLANA, 'Sin mirar: plana'
    else:
        x, z, b_ = H[i]
        t_ = (f'x = {coma(x)} m, {"«hueco»" if z else "«pared»"}: a ±0,3 m, el {coma(100*TABLA_H[i][2], 0)} %'
              f'   (ancha {coma(100*TABLA_H[i][3], 0)} %, estrecha {coma(100*TABLA_H[i][4], 0)} %)')
        ax.axvline(x, color=IQS_VERDE, lw=2)
    ax.stairs(b_, BORDES, fill=True, color=IQS_AZUL)
    pasillo(ax, rotulos=ax is axs[0])
    ax.set_ylim(0, max(b_.max() * 1.35, 0.03)); ax.set_ylabel('p', fontsize=9)
    ax.set_title(t_, fontsize=10, loc='left', pad=14 if ax is axs[0] else 4)
ax = axs[-1]                                       # la creencia cerca de la verdad, paso a paso
for a, b_, _ in PUERTAS:
    ax.axvspan(a, b_, color=GRIS, alpha=0.22, lw=0)
if CERCA_ANCHA is not None:
    ax.plot(CERCA_ANCHA[:, 0], CERCA_ANCHA[:, 1], '-', color=GRIS, lw=1.5, label='desde la ancha (0,95 m)')
ax.plot(CERCA[:, 0], CERCA[:, 1], '-', color=IQS_AZUL, lw=2, label=f'desde {coma(X0)} m')
h_ = CERCA[:, 2] > 0
ax.plot(CERCA[h_, 0], CERCA[h_, 1], 'o', color=IQS_AZUL, ms=6)
ax.plot(CERCA[~h_, 0], CERCA[~h_, 1], 'o', mfc='w', mec=IQS_AZUL, ms=5)
ax.axhline(70, color='k', ls='--', lw=1)
ax.set_ylim(0, 105); ax.set_xlim(0, 13); ax.set_xticks(range(0, 14)); ejes(ax)
ax.set_ylabel('cerca de la verdad (%)'); ax.set_xlabel('x a lo largo del pasillo (m)')
ax.set_title('La creencia a ±0,3 m de la verdad, en cada paso', fontsize=10, loc='left')
ax.legend(handles=ax.get_legend_handles_labels()[0] + [
    Line2D([], [], color=IQS_VERDE, lw=2, label='la x verdadera'),
    Line2D([], [], color=IQS_AZUL, marker='o', ls='none', label='«hueco»'),
    Line2D([], [], color=IQS_AZUL, marker='o', mfc='w', ls='none', label='«pared»'),
    Line2D([], [], color='k', ls='--', lw=1, label='70 %')], loc='upper center', bbox_to_anchor=(0.5, -0.38), fontsize=9, ncol=6,
    frameon=False)
plt.show()

**Qué tiene que salir:** desde 0,95 m, a ±0,3 m: 12,9 % al empezar, 79,9 % en 2,45 m y 97,4 % tras la estrecha.

## 3. Kalman en una dimensión, por el mismo pasillo

*En clase · diapositivas 54 y 55*

Predecir suma varianzas; corregir reparte con L = (σ⁻)²/((σ⁻)² + r). El robot cree estar en 5,00 m con 10 cm; una baliza dice 5,12 m con 5 cm.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def kalman_1d(x, s2, delta, q, z, r):
    """Un paso de Kalman en una dimension. Predecir: x̂⁻ = x̂ + δ y (σ⁻)² = σ² + q (las varianzas se suman).
    Corregir con la medida z: L = (σ⁻)² / ((σ⁻)² + r); x̂ = x̂⁻ + L·(z − x̂⁻); σ² = (1 − L)·(σ⁻)²."""
    x_, s2_ = x + delta, s2 + q
    L = s2_ / (s2_ + r)
    return x_ + L * (z - x_), (1 - L) * s2_, L, s2_


def pasillo_kalman(q=0.0018, r=0.05**2, s0=0.10, pasos=24, semilla=25):
    """El robot recorre el pasillo en pasos de 0,5 m (la odometria, con σ = 4,2 cm por paso: q = 0,0018 m²) y en
    cada paso mide su x con σ = 5 cm (r = 0,0025 m²). Devuelve la verdad, la odometria sola, la medida sola
    y Kalman, con su σ y su L."""
    rng = np.random.default_rng(semilla)
    x = xo = xk = 0.5
    s2 = s0**2
    filas = []
    for k in range(pasos):
        d = 0.5 + rng.normal(0, np.sqrt(q))      # lo que dice la odometria; el robot avanza 0,5 m
        x += 0.5
        z = x + rng.normal(0, np.sqrt(r))        # la medida
        xo += d                                  # a estima: solo la odometria
        xk, s2, L, s2_ = kalman_1d(xk, s2, d, q, z, r)
        filas.append((x, xo, z, xk, np.sqrt(s2_), np.sqrt(s2), L))
    return np.array(filas)


def l_estacionaria(q, r, n=500):
    """La ganancia a la que llega Kalman cuando q y r no cambian: la de un filtro de reparto fijo."""
    s2 = 1.0
    for _ in range(n):
        s2_ = s2 + q
        L = s2_ / (s2_ + r)
        s2 = (1 - L) * s2_
    return L

X_C, S2_C, L_C, _ = kalman_1d(5.00, 0.10**2, 0.0, 0.0, 5.12, 0.05**2)   # la corrección de la diapositiva 55
SIGMA_C = np.sqrt(S2_C)
# para la figura: las tres campanas (la predicha, la de la baliza y la corregida) [densidad, 1/m]
XS = np.linspace(4.6, 5.35, 400)
campana = lambda m, s: np.exp(-(XS - m)**2 / (2 * s**2)) / (s * np.sqrt(2 * np.pi))
G_PRED, G_BALIZA, G_CORR = campana(5.00, 0.10), campana(5.12, 0.05), campana(X_C, SIGMA_C)

In [ ]:
# @title ✔ Resultado
print(f'L = 0,01/(0,01 + 0,0025) = {coma(L_C)} ;  x̂ = {coma(X_C,3)} m ;  σ = {coma(SIGMA_C*100)} cm  (menos que 10 y que 5)')

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(XS, G_PRED, color=GRIS, lw=2, label='predicha: 5,00 m, σ = 10 cm')
ax.plot(XS, G_BALIZA, color=AMBAR, lw=2, label='la baliza: 5,12 m, σ = 5 cm')
ax.plot(XS, G_CORR, color=IQS_AZUL, lw=2.6, label=f'corregida: {coma(X_C, 3)} m, σ = {coma(SIGMA_C*100)} cm')
ax.annotate('', (X_C, G_CORR.max() * 1.07), (5.00, G_CORR.max() * 1.07), arrowprops=dict(arrowstyle='->', color=IQS_AZUL, lw=1.5))
ax.text(5.0, G_CORR.max() * 1.11, f'L·(z − x̂⁻) = {coma(L_C, 1)} · 12 cm', fontsize=9.5, color=IQS_AZUL, va='bottom')
ax.set_xlim(4.6, 5.35); ax.set_ylim(0, G_CORR.max() * 1.3); ejes(ax)
ax.set_xlabel('x (m)'); ax.set_ylabel('densidad (1/m)'); ax.legend(loc='upper left', fontsize=9.5, frameon=False)
ax.set_title(f'Una corrección: la ganancia L = {coma(L_C, 1)} reparte entre la predicha y la baliza', fontsize=11)
plt.show()

**Qué tiene que salir:** L = 0,80, x̂ = 5,096 m y σ = 4,47 cm.

### En clase: la baliza de 20 cm

Poned `R_BALIZA = 0.20`: ¿a qué L llega, y cuánto se equivoca el filtro?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# σ de la medida de la baliza [m]: probad 0.20
R_BALIZA = 0.05  # @param {type:"number"}
# varianza de la odometría en un paso de 0,5 m [m²]
Q_PASO = 0.0018  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
F = pasillo_kalman(Q_PASO, R_BALIZA**2, s0=0.10, pasos=24, semilla=25)
rmse = lambda e: np.sqrt((e**2).mean())
RMSE_ESTIMA, RMSE_MEDIDA, RMSE_KALMAN = rmse(F[:, 1] - F[:, 0]), rmse(F[:, 2] - F[:, 0]), rmse(F[:, 3] - F[:, 0])
# para la figura: el paso, el error de cada uno frente a la verdad [cm] y, si vuestra baliza es otra, la L con la de 5 cm
K_PASO = np.arange(1, len(F) + 1)
ERR_ESTIMA, ERR_MEDIDA, ERR_KALMAN = ((F[:, j] - F[:, 0]) * 100 for j in (1, 2, 3))
BANDA_2S = 2 * F[:, 5] * 100                     # ±2σ de Kalman, tras corregir [cm]
L_5CM = None if R_BALIZA == 0.05 else pasillo_kalman(Q_PASO, 0.05**2, s0=0.10, pasos=24, semilla=25)[:, 6]

In [ ]:
# @title ✔ Resultado
print('L paso a paso:', ', '.join(coma(v, 3) for v in F[:6, 6]), '...', coma(F[-1, 6], 3))
print(f'σ corregida al final: {coma(F[-1, 5]*100)} cm')
print(f'Error medio (RMSE) en los 24 pasos: a estima {coma(RMSE_ESTIMA*100,1)} cm ; la medida sola {coma(RMSE_MEDIDA*100,1)} cm ;'
      f' Kalman {coma(RMSE_KALMAN*100,1)} cm')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw=dict(width_ratios=[1.6, 1], wspace=0.22))
a1.axhline(0, color=IQS_VERDE, lw=2, label='la verdad')
a1.fill_between(K_PASO, -BANDA_2S, BANDA_2S, color=IQS_AZUL, alpha=0.15, lw=0, label='±2σ de Kalman')
a1.plot(K_PASO, ERR_ESTIMA, '--', color=ROJO, lw=1.8, label=f'la odometría sola: {coma(RMSE_ESTIMA*100, 1)} cm')
a1.plot(K_PASO, ERR_MEDIDA, 'o', color=AMBAR, ms=4, label=f'la medida sola: {coma(RMSE_MEDIDA*100, 1)} cm')
a1.plot(K_PASO, ERR_KALMAN, '-', color=IQS_AZUL, lw=2, label=f'Kalman: {coma(RMSE_KALMAN*100, 1)} cm')
a1.set_xlim(0.5, len(F) + 0.5); a1.set_xlabel('paso de 0,5 m'); a1.set_ylabel('error frente a la verdad (cm)'); ejes(a1)
a1.legend(loc='upper left', fontsize=9, ncol=3, frameon=True, framealpha=0.9, bbox_to_anchor=(0, -0.17))
a1.set_title(f'Con la baliza de {coma(R_BALIZA*100, 0)} cm: el error de cada uno y su RMSE', fontsize=11)
if L_5CM is not None:
    a2.plot(K_PASO, L_5CM, 'o-', color=GRIS, lw=1.5, ms=4, label=f'baliza de 5 cm: {coma(L_5CM[-1], 2)}')
a2.plot(K_PASO, F[:, 6], 'o-', color=IQS_AZUL, lw=2, ms=4, label=f'baliza de {coma(R_BALIZA*100, 0)} cm: {coma(F[-1, 6], 2)}')
a2.set_ylim(0, 1); a2.set_xlim(0.5, len(F) + 0.5); a2.set_xlabel('paso de 0,5 m'); a2.set_ylabel('L'); ejes(a2)
a2.legend(loc='upper right', fontsize=9, frameon=False); a2.set_title('La ganancia, paso a paso', fontsize=11)
plt.show()

**Qué tiene que salir:** con 5 cm, L de 0,825 a 0,562; error medio, 14,0 cm a estima, 4,8 la medida y 3,7 Kalman.

### ✎ Vuestra respuesta (tres líneas)

*Doble clic aquí y escribid, con cifras. Se sube al acabar la clase, con el S18b.*

1. La puerta: cuántas «abierta» seguidas para pasar de 0,99, con el detector de la tabla y con uno de 0,95 y 0,05.
2. El pasillo desde x = 5,95 m: cuándo queda un solo pico, y por qué tarda más que desde la ancha.
3. Kalman con la baliza de 5 y de 20 cm: a qué L llega, y cuánto se equivoca.

## 4. Kalman, ya visto: el complementario de S6 y el observador de S15 (en casa)

*En casa · diapositiva 58*

Con q y r fijos, la L de régimen solo depende de q/r. S6: θ̂ = θ̂⁻ + 0,02·(θ_acel − θ̂⁻), Kalman con L = 0,02. S15: dx̂/dt = A·x̂ + B·V + ℓ·(q − C·x̂), con ℓ de los polos.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los casos: (nombre, q, r), las varianzas del proceso y de la medida [m² o rad², en las mismas unidades]
CASOS = [('el 98/2 de S6', 4.08e-4, 1.0), ('baliza de 20 cm', 0.0018, 0.04), ('el pasillo', 0.0018, 0.0025), ('baliza de 1 cm', 0.0018, 0.0001)]  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
L_CASOS = [l_estacionaria(q, r) for _, q, r in CASOS]      # la L estacionaria de cada caso
# para la figura: la curva de la L estacionaria según q/r, y la L de cada caso paso a paso, desde σ² = 1
QR = np.logspace(-5, 2, 200)
L_CURVA = [l_estacionaria(v, 1.0) for v in QR]


def l_paso_a_paso(q, r, n=500):
    """La L de cada paso, con las cuentas de l_estacionaria."""
    s2, Ls = 1.0, []
    for _ in range(n):
        s2_ = s2 + q
        Ls.append(s2_ / (s2_ + r))
        s2 = (1 - Ls[-1]) * s2_
    return np.array(Ls)

L_PASOS = [l_paso_a_paso(q, r) for _, q, r in CASOS]

In [ ]:
# @title ✔ Resultado
for (nombre, q, r), L_ in zip(CASOS, L_CASOS):
    qr_ = cien(q / r) if q / r < 0.01 else coma(q / r, 3 if q / r < 1 else 1)
    print(f'{nombre:16s} q/r = {qr_:>9s}  ->  L estacionaria = {coma(L_, 3)}')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw=dict(wspace=0.2, right=0.84))
a1.semilogx(QR, L_CURVA, color=IQS_AZUL, lw=2.2)
for (nombre, q, r), L_ in zip(CASOS, L_CASOS):
    a1.plot(q / r, L_, 'o', color='k', ms=7, zorder=5)
    if np.log10(q / r) < -1.5:                    # a la izquierda, donde la curva va pegada al cero: encima de ella
        a1.text(1.3e-5, L_ + 0.05, f'{nombre}: {coma(L_, 2)}', va='bottom', fontsize=9.5)
    else:
        a1.annotate(f'{nombre}: {coma(L_, 2)}', (q / r, L_), (-8, 4), textcoords='offset points', ha='right', va='bottom', fontsize=9.5)
a1.set_ylim(0, 1.05); a1.set_xlim(1e-5, 1e2); a1.yaxis.set_major_formatter(COMA); a1.xaxis.set_minor_locator(plt.NullLocator())
a1.set_xlabel('q / r: el ruido del proceso entre el de la medida'); a1.set_ylabel('L estacionaria')
a1.set_title('La L a la que llega Kalman', fontsize=11)
for (nombre, _, _), Ls in zip(CASOS, L_PASOS):
    a2.semilogx(np.arange(1, len(Ls) + 1), Ls, lw=2.2 if nombre == CASOS[0][0] else 1.3,
                color=IQS_AZUL if nombre == CASOS[0][0] else GRIS)
    a2.text(1.02, Ls[-1], f'{nombre}: {coma(Ls[-1], 2)}', transform=a2.get_yaxis_transform(), va='center', fontsize=9.5,
            color=IQS_AZUL if nombre == CASOS[0][0] else 'k')
a2.set_ylim(0, 1.05); a2.set_xlim(1, len(L_PASOS[0])); a2.yaxis.set_major_formatter(COMA)
a2.xaxis.set_minor_locator(plt.NullLocator())
a2.set_xlabel('paso, desde σ² = 1 (escala logarítmica)'); a2.set_ylabel('L')
a2.set_title('Paso a paso, cada L se queda en la suya', fontsize=11)
plt.show()

**Qué tiene que salir:** L = 0,020, 0,191, 0,562 y 0,950.

### Ejercicio 1 (en casa): una puerta con dos medidas

Un detector de 0,8 y 0,4 dice «abierta» dos veces, desde 0,5: calculad a mano la creencia, con η. ¿Qué hipótesis usáis al encadenar, y cuándo falla en el almacén?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el detector: p(«abierta» | abierta) y p(«abierta» | cerrada) [de 0 a 1]
E1_ACIERTO = 0.8  # @param {type:"number"}
E1_FALSA = 0.4  # @param {type:"number"}
# la creencia antes de mirar, p(abierta) [de 0 a 1], y lo que dice el detector
E1_PREVIA = 0.5  # @param {type:"number"}
E1_MEDIDAS = ['abierta', 'abierta']  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E1_CREENCIAS, E1_ETAS = [E1_PREVIA], []        # p(abierta) sin mirar y tras cada medida, y η de cada una
for e1_z in E1_MEDIDAS:
    e1_b, e1_eta = bayes_puerta(E1_CREENCIAS[-1], e1_z, E1_ACIERTO, E1_FALSA)
    E1_CREENCIAS.append(e1_b); E1_ETAS.append(e1_eta)

In [ ]:
# @title ✔ Resultado
for e1_i, (e1_z, e1_eta, e1_b) in enumerate(zip(E1_MEDIDAS, E1_ETAS, E1_CREENCIAS[1:]), 1):
    print(f'{e1_i}.ª «{e1_z}»: η = {coma(e1_eta, 3)} ; p(abierta) = {coma(e1_b, 3)}')

fig, ax = plt.subplots(figsize=(max(5, 2.5 + 0.9 * len(E1_CREENCIAS)), 3))
ax.bar(range(len(E1_CREENCIAS)), E1_CREENCIAS, width=0.55, color=[IQS_AZUL] + [IQS_AZUL if z == 'abierta' else AMBAR for z in E1_MEDIDAS])
for e1_i, e1_b in enumerate(E1_CREENCIAS):
    ax.text(e1_i, e1_b + 0.03, coma(e1_b, 3), ha='center', fontsize=10, color=IQS_AZUL)
ax.set_xticks(range(len(E1_CREENCIAS)), ['sin mirar'] + [f'«{z}»' for z in E1_MEDIDAS])
ax.set_ylim(0, 1.15); ax.set_ylabel('p(abierta)'); ax.yaxis.set_major_formatter(COMA); ax.grid(axis='x')
ax.set_title(f'El detector de {coma(E1_ACIERTO, 1)} y {coma(E1_FALSA, 1)}', fontsize=11)
plt.show()

### Ejercicio 2 (en casa): un detector peor

¿En cuántas de 100 semillas acaba localizado el robot (más del 80 % a ±0,3 m), con el detector de 0,8 y 0,1 y con el de 0,6 y 0,2? ¿Por qué?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los detectores laterales que se comparan: (p(«hueco» | hueco), p(«hueco» | pared)) [de 0 a 1]
E2_DETECTORES = [(0.8, 0.1), (0.6, 0.2)]  # @param {type:"raw"}
# cuántas semillas, de dónde sale [m] y cuántos pasos de 0,5 m (19: hasta x = 9,95 m, tras la estrecha)
E2_SEMILLAS = 100  # @param {type:"integer"}
E2_X0 = 0.95  # @param {type:"number"}
E2_PASOS = 19  # @param {type:"integer"}
# localizado: más de esta creencia a ±0,3 m de la verdad, al final [de 0 a 1]
E2_UMBRAL = 0.8  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E2 = []                                          # (detector, la creencia final a ±0,3 m en cada semilla, cuántas pasan)
for e2_det in E2_DETECTORES:
    e2_m = []
    for e2_s in range(int(E2_SEMILLAS)):
        e2_h = recorre(*e2_det, semilla=e2_s, x0=E2_X0, pasos=int(E2_PASOS))
        e2_m.append(masa_cerca(e2_h[-1][2], e2_h[-1][0]))
    E2.append((e2_det, np.array(e2_m), int((np.array(e2_m) > E2_UMBRAL).sum())))

In [ ]:
# @title ✔ Resultado
for e2_det, e2_m, e2_ok in E2:
    print(f'detector {coma(e2_det[0], 1)} / {coma(e2_det[1], 1)}: {e2_ok} de {len(e2_m)} acaban con más del {coma(100*E2_UMBRAL, 0)} % a ±0,3 m')

fig, axs = plt.subplots(1, len(E2), figsize=(min(15, 5.5 * len(E2)), 3.2), sharey=True, squeeze=False)
for ax, (e2_det, e2_m, e2_ok) in zip(axs[0], E2):
    ax.hist(100 * e2_m, bins=np.arange(0, 105, 5), color=IQS_AZUL)
    ax.axvline(100 * E2_UMBRAL, color='k', ls='--', lw=1)
    ax.set_xlim(0, 100); ax.set_xlabel('creencia final a ±0,3 m de la verdad (%)'); ejes(ax)
    ax.set_title(f'Detector {coma(e2_det[0], 1)} y {coma(e2_det[1], 1)}: {e2_ok} de {len(e2_m)} pasan del {coma(100*E2_UMBRAL, 0)} %', fontsize=11)
axs[0, 0].set_ylabel('semillas')
plt.show()

### Ejercicio 3 (en casa): el 98/2 de S6, como Kalman

El 98/2 es L = 0,02: ¿cuántas veces más ruidoso, en σ, es el acelerómetro que un paso del giróscopo? ¿Qué α y qué τ, a 100 Hz, daría uno 10 veces más ruidoso? En régimen, L²/(1 − L) = q/r y τ = α·Δt/(1 − α).

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el reparto del complementario de S6: la L del 98/2 [de 0 a 1]
E3_L_S6 = 0.02  # @param {type:"number"}
# el otro acelerómetro: cuántas veces más ruidoso, en σ, que un paso del giróscopo [veces]
E3_OTRO = 10  # @param {type:"number"}
# el periodo de muestreo de S6 [s]: 100 Hz
E3_DT = 0.01  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e3_qr = E3_L_S6**2 / (1 - E3_L_S6)               # en régimen, L²/(1 − L) = q/r
e3_veces = 1 / np.sqrt(e3_qr)                    # q/r = (σ_paso/σ_acel)²: las veces, en σ
E3 = []                                          # (veces, q/r, L, α, τ [s]): el 98/2 (L vuelve a salir 0,02) y el otro
for e3_v in (e3_veces, E3_OTRO):
    e3_q = 1 / e3_v**2
    e3_L = l_estacionaria(e3_q, 1.0)
    E3.append((e3_v, e3_q, e3_L, 1 - e3_L, (1 - e3_L) * E3_DT / e3_L))   # α = 1 − L y τ = α·Δt/(1 − α)
# para la figura: τ según las veces, de 1 a 300
E3_CURVA_V = np.logspace(0, np.log10(300), 60)
E3_CURVA_TAU = [(1 - e3_l) * E3_DT / e3_l for e3_l in (l_estacionaria(1 / e3_v**2, 1.0) for e3_v in E3_CURVA_V)]

In [ ]:
# @title ✔ Resultado
print(f'98/2: L = {coma(E3_L_S6)} -> q/r = L²/(1 − L) = {cien(e3_qr)} -> σ_acel/σ_paso = 1/√(q/r) = {coma(e3_veces, 1)} veces')
for e3_v, e3_q, e3_L, e3_a, e3_tau in E3:
    print(f'{coma(e3_v, 1)} veces: q/r = {cien(e3_q)} ; L = {coma(e3_L, 4)} ; α = {coma(e3_a, 4)} ; τ = {coma(e3_tau, 3)} s')

fig, ax = plt.subplots(figsize=(8, 3.3))
ax.loglog(E3_CURVA_V, E3_CURVA_TAU, color=IQS_AZUL, lw=2)
for e3_v, e3_q, e3_L, e3_a, e3_tau in E3:
    ax.plot(e3_v, e3_tau, 'o', color='k', ms=7, zorder=5)
    ax.annotate(f'{coma(e3_v, 1)} veces: {coma(100*e3_a, 1)}/{coma(100*e3_L, 1)}, τ = {coma(e3_tau, 3)} s', (e3_v, e3_tau),
                (-8, 6), textcoords='offset points', ha='right', va='bottom', fontsize=9.5)
ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)
ax.xaxis.set_minor_locator(plt.NullLocator()); ax.yaxis.set_minor_locator(plt.NullLocator())
ax.set_xlabel('σ del acelerómetro / σ de un paso del giróscopo'); ax.set_ylabel('τ (s)')
ax.set_title(f'El complementario que sale de los dos ruidos, a {coma(1/E3_DT, 0)} Hz', fontsize=11)
plt.show()

---

## Soluciones

**Ejercicio 1.** p(«abierta») = 0,8·0,5 + 0,4·0,5 = 0,6: η = 1,667 y p(abierta) = **0,667**. La segunda: 0,533/0,667 = **0,800** (η = 1,5). Al encadenar se usa que las medidas son independientes entre sí si se conoce el estado. Falla si alguien está parado en la puerta: el detector se equivoca dos veces por la misma causa. Se arregla metiendo esa causa en el estado, no ajustando el ruido.

**Ejercicio 2.** Con 0,8 y 0,1, 68 de 100; con 0,6 y 0,2, solo 5. Cada «hueco» multiplica las celdas de hueco frente a las de pared por 0,8/0,1 = 8 con el bueno y por 0,6/0,2 = 3 con el malo; cada «pared», por 0,2/0,9 = 0,22 frente a 0,4/0,8 = 0,5. Con 3 y 0,5, las medidas de las dos puertas no bastan para separar las hipótesis.

**Ejercicio 3.** 49,5 veces: q/r = 4,08·10⁻⁴ y L = 0,0200. Con 10 veces, q/r = 0,01, L = 0,0951 y α = 0,905: 90,5/9,5, con τ = 0,095 s en vez de 0,49 s. El 98/2 sale de los dos ruidos.

---

## Para llevarse de esta sesión

- **Bayes da la vuelta:** de p(z | x), el sensor, a p(x | z), lo que se cree.
- **Se encadena si las medidas son independientes conociendo el estado.**
- **Predecir ensancha; corregir estrecha.**
- **Kalman reparte con la ganancia:** fusionar da menos error que cada uno solo.
- **El complementario de S6 y el observador de S15 eran Kalman.**

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*